
# Sistema de Recomendação de Filmes com IMDb

Este notebook implementa um sistema simples de recomendação de filmes utilizando:

- Dataset IMDb
- TF-IDF
- Similaridade de Cosseno
- Recomendação baseada em conteúdo

O objetivo é recomendar filmes similares com base em:
- gênero
- descrição
- palavras-chave


In [ ]:

# Instalação das bibliotecas (descomente se necessário)
# !pip install pandas scikit-learn numpy matplotlib


In [1]:

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')



## Carregando os dados

O notebook espera um CSV contendo pelo menos:

- title
- genres
- overview

Você pode usar datasets do:
- Kaggle
- IMDb
- TMDB

Exemplo:
`imdb_movies.csv`


In [6]:

# Caminho do dataset
DATASET_PATH = '../DATABASES/imdb_movies.csv'

# Carregamento
df = pd.read_csv(DATASET_PATH)

# Visualização inicial
df.head(10)


,title,genres,overview
0,The Dark Knight,Action Crime Drama,"Batman faces the Joker, a criminal mastermind ..."
1,Inception,Action Sci-Fi Thriller,A thief enters people's dreams to steal secret...
2,Interstellar,Adventure Drama Sci-Fi,Astronauts travel through a wormhole in search...
3,The Matrix,Action Sci-Fi,A hacker discovers reality is a simulation con...
4,Pulp Fiction,Crime Drama,The lives of criminals intertwine in a series ...
5,Fight Club,Drama,An insomniac office worker forms an undergroun...
6,Forrest Gump,Drama Romance,A simple man witnesses and influences major hi...
7,The Shawshank Redemption,Drama,Two imprisoned men form a friendship over deca...
8,The Godfather,Crime Drama,The aging patriarch of a mafia family transfer...
9,Gladiator,Action Adventure Drama,A Roman general seeks revenge after betrayal b...


In [4]:

# Selecionando colunas relevantes

columns = ['title', 'genres', 'overview']

df = df[columns]

# Removendo valores nulos
df = df.dropna()

df.head()


,title,genres,overview
0,The Dark Knight,Action Crime Drama,"Batman faces the Joker, a criminal mastermind ..."
1,Inception,Action Sci-Fi Thriller,A thief enters people's dreams to steal secret...
2,Interstellar,Adventure Drama Sci-Fi,Astronauts travel through a wormhole in search...
3,The Matrix,Action Sci-Fi,A hacker discovers reality is a simulation con...
4,Pulp Fiction,Crime Drama,The lives of criminals intertwine in a series ...



## Pré-processamento

Vamos combinar:
- gêneros
- descrição

em um único campo textual.


In [8]:

# Combinação das features textuais
df['content'] = df['genres'] + ' ' + df['overview']

df[['title', 'content']].head()


,title,content
0,The Dark Knight,"Action Crime Drama Batman faces the Joker, a c..."
1,Inception,Action Sci-Fi Thriller A thief enters people's...
2,Interstellar,Adventure Drama Sci-Fi Astronauts travel throu...
3,The Matrix,Action Sci-Fi A hacker discovers reality is a ...
4,Pulp Fiction,Crime Drama The lives of criminals intertwine ...



## Vetorização TF-IDF


In [9]:

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['content'])

print('Formato da matriz TF-IDF:', tfidf_matrix.shape)


Formato da matriz TF-IDF: (20, 132)



## Similaridade de Cosseno


In [10]:

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cosine_sim.shape)


(20, 20)



## Índice de Filmes


In [11]:

indices = pd.Series(df.index, index=df['title']).drop_duplicates()

indices.head()


title
The Dark Knight    0
Inception          1
Interstellar       2
The Matrix         3
Pulp Fiction       4
dtype: int64


# Função de Recomendação


In [12]:

def recomendar_filmes(titulo, quantidade=10):
    if titulo not in indices:
        return f'Filme "{titulo}" não encontrado no dataset.'

    idx = indices[titulo]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:quantidade+1]

    movie_indices = [i[0] for i in sim_scores]

    resultados = df[['title', 'genres']].iloc[movie_indices].copy()

    resultados['similaridade'] = [round(i[1], 3) for i in sim_scores]

    return resultados



# Exemplo de Uso


In [14]:

recomendar_filmes('Interstellar', quantidade=5)


,title,genres,similaridade
19,Star Wars,Action Adventure Sci-Fi,0.144
12,The Avengers,Action Adventure Sci-Fi,0.136
3,The Matrix,Action Sci-Fi,0.116
1,Inception,Action Sci-Fi Thriller,0.099
9,Gladiator,Action Adventure Drama,0.061



# Conclusão

Este notebook demonstrou um sistema de recomendação de filmes baseado em conteúdo usando técnicas clássicas de Machine Learning.

O sistema recomenda filmes similares analisando características textuais e calculando similaridade entre eles.
